## I. Setup Software and Some Libraries

We first need to set the working environment and the path to the dataset.

In [1]:
import sys

SOFTWARE_DIR = '/mnt/data/spine_bilal/spine/' # Change this path to your software install
DATA_DIR = '/mnt/data/polaris/2025_02_17/inference/mc_handscan/' # Change this path if you are not on SDF (see main README)

# SOFTWARE_DIR = '/home/bilal/spine_bilal/spine/'  # Adjust if needed
# DATA_DIR = '/home/bilal/inference/mc_handscan/'

# Set software directory
sys.path.append(SOFTWARE_DIR)

import numpy as np
import math
import pandas as pd
from collections import OrderedDict

from scipy.spatial.distance import cdist

Now pass the analysis configuration.

In [2]:
import yaml
from spine.driver import Driver


DATA_PATH = DATA_DIR + 'MiniRun6.1_1E19_RHC.spine.0000000.MLRECO_SPINE.hdf5'

cfg = '''
# Load HDF5 files
io:
  reader:
    name: hdf5
    file_keys: DATA_PATH
    skip_unknown_attrs: true
# Build reconstruction output representations
build:
  mode: both
  units: cm
  fragments: false
  particles: true
  interactions: true
'''.replace('DATA_PATH', DATA_PATH)

cfg = yaml.safe_load(cfg)
driver = Driver(cfg)

Welcome to JupyROOT 6.30/06

 ██████████   ██████████    ███   ███       ██   ███████████
███        █  ██       ███   █    █████     ██   ██         
  ████████    ██       ███  ███   ██  ████  ██   ██████████ 
█        ███  ██████████     █    ██     █████   ██         
 ██████████   ██            ███   ██       ███   ███████████

Release version: 0.2.2

$CUDA_VISIBLE_DEVICES=

Configuration processed at: Linux bd9fe1b7ae37 5.15.167.4-microsoft-standard-WSL2 #1 SMP Tue Nov 5 00:21:55 UTC 2024 x86_64 x86_64 x86_64 GNU/Linux

base: {seed: 1742322738}
io:
  reader: {name: hdf5, file_keys: /mnt/data/polaris/2025_02_17/inference/mc_handscan/MiniRun6.1_1E19_RHC.spine.0000000.MLRECO_SPINE.hdf5,
    skip_unknown_attrs: true}
build: {mode: both, units: cm, fragments: false, particles: true, interactions: true}

Will load 1 file(s):
  - /mnt/data/polaris/2025_02_17/inference/mc_handscan/MiniRun6.1_1E19_RHC.spine.0000000.MLRECO_SPINE.hdf5

Total number of entries in the file(s): 172

Total numb

Map ND-LAr events to their corresponding `entry` numbers in SPINE files using the `run_info` field.

In [3]:
print('Number of entries in the loader:', len(driver))

import h5py

# Target Event (Update this as needed)
target_event = 79 # Example: Target Event Number

# Initialize entry as None
ENTRY = None

# Locate Entry Corresponding to Target Event
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    for idx, (run, subrun, event) in enumerate(run_info):
        if event == target_event:
            ENTRY = idx
            print(f"Found target event {target_event} at entry {idx}")
            break

if ENTRY is None:
    print(f"Target event {target_event} not found.")

Number of entries in the loader: 172
Found target event 79 at entry 72


Now initialize the driver with this configuration.

In [4]:
if ENTRY is not None:
    data = driver.process(entry=ENTRY)
    print(f"Processing data for entry: {ENTRY}")
else:
    print("No valid entry found. Ensure previous Cell has run successfully.")

Processing data for entry: 72


Print reference list of PIDs and their corresponding particle types

In [5]:
from spine.utils.globals import PID_LABELS

PID_LABELS

{-1: 'Unknown',
 0: 'Photon',
 1: 'Electron',
 2: 'Muon',
 3: 'Pion',
 4: 'Proton',
 5: 'Kaon'}

Retrieves and analyzes the `reco/truth_particles` and `reco/truth_interactions` data structures. It provides a count of each structure and collects the unique **Truth and Reco Interaction IDs** from the `reco/truth_particles` dataset.

In [6]:
# Retrieving data structures
reco_particles = data['reco_particles']
truth_particles = data['truth_particles']
reco_interactions = data['reco_interactions']
truth_interactions = data['truth_interactions']

# Print counts of different data structures
print(f"Reco Particles: {len(reco_particles)}")
print(f"Truth Particles: {len(truth_particles)}")
print(f"Reco Interactions: {len(reco_interactions)}")
print(f"Truth Interactions: {len(truth_interactions)}")


# Collect Reco and Truth Interaction IDs
reco_interaction_ids = set(particle.interaction_id for particle in reco_particles)
truth_interaction_ids = set(particle.interaction_id for particle in truth_particles)

print(f"Reco Interaction IDs: {reco_interaction_ids}")  # Sorted for better readability
print(f"Truth Interaction IDs: {truth_interaction_ids}")  # Sorted for better readability

# Count the number of reco particles associated with each reco interaction ID
reco_interaction_particle_counts = {}
for particle in reco_particles:
    interaction_id = particle.interaction_id
    if interaction_id not in reco_interaction_particle_counts:
        reco_interaction_particle_counts[interaction_id] = 0
    reco_interaction_particle_counts[interaction_id] += 1

# Count the number of truth particles associated with each truth interaction ID
truth_interaction_particle_counts = {}
for particle in truth_particles:
    interaction_id = particle.interaction_id
    if interaction_id not in truth_interaction_particle_counts:
        truth_interaction_particle_counts[interaction_id] = 0
    truth_interaction_particle_counts[interaction_id] += 1

# Print the number of particles for each interaction ID
print("\nNumber of Reco Particles associated with each Reco Interaction ID:")
for interaction_id, count in reco_interaction_particle_counts.items():
    print(f"Reco Interaction ID {interaction_id}: {count} particles")

print("\nNumber of Truth Particles associated with each Truth Interaction ID:")
for interaction_id, count in truth_interaction_particle_counts.items():
    print(f"Truth Interaction ID {interaction_id}: {count} particles")

Reco Particles: 17
Truth Particles: 18
Reco Interactions: 5
Truth Interactions: 5
Reco Interaction IDs: {0, 1, 2, 3, 4}
Truth Interaction IDs: {0, 1, 2, 3, 4}

Number of Reco Particles associated with each Reco Interaction ID:
Reco Interaction ID 0: 2 particles
Reco Interaction ID 3: 9 particles
Reco Interaction ID 1: 2 particles
Reco Interaction ID 4: 2 particles
Reco Interaction ID 2: 2 particles

Number of Truth Particles associated with each Truth Interaction ID:
Truth Interaction ID 0: 4 particles
Truth Interaction ID 1: 2 particles
Truth Interaction ID 3: 1 particles
Truth Interaction ID 2: 9 particles
Truth Interaction ID 4: 2 particles


## II. Filtering Reconstructed Particles for Target Interaction ID

The script locates the target reconstructed interaction ID (`target_reco_interaction_id`) in the dataset, retrieves its entry, and filters all **reconstructed particles** and the matched truth particles that belong to the specified interaction ID.

It prints details of reconstructed and truth particles and their interaction IDs, along with the total count of filtered particles.

In [7]:
import h5py

# Define the target reconstructed interaction ID
target_reco_interaction_id = 2  # Example: Target Interaction ID

# Define spatial limits
distance_from_wall = 5.0
minX = -63.931 + distance_from_wall
maxX = +63.931 - distance_from_wall
minY = -62.076 + distance_from_wall
maxY = +62.076 - distance_from_wall
minZ = -64.538 + distance_from_wall
maxZ = +64.538 - distance_from_wall

# Locate the entry corresponding to the target event
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    target_entry = None
    
    for idx, (run, subrun, event) in enumerate(run_info):
        if event == target_event:
            target_entry = idx
            print(f"Found target event {target_event} at entry {idx}")
            break
    else:
        print(f"Event {target_event} not found in run_info.")
        target_entry = None

# Process the entry if found
if target_entry is not None:
    data = driver.process(entry=target_entry)
    reco_particles = data['reco_particles']
    print(f"Total Reconstructed Particles for Event {target_event}: {len(reco_particles)}")
    
    # Filter and print particles within limits, primary, and belonging to the target interaction ID
    filtered_primary_particles = []
    for i, particle in enumerate(reco_particles):
        # Access start_point for spatial coordinates using dot notation
        start_point = particle.start_point
        x, y, z = start_point
        
        # Check if the particle is primary and matches the target interaction ID
        is_primary = particle.is_primary
        interaction_id = particle.interaction_id  # Get the interaction ID
        
        # Apply spatial limits, primary condition, and target interaction ID
        if (
            # is_primary and
            # minX < x < maxX and
            # minY < y < maxY and
            # minZ < z < maxZ and
            interaction_id == target_reco_interaction_id
        ):
            filtered_primary_particles.append(particle)
            print(f"reco_particles[{i}]: {particle} | Reco Interaction ID: {interaction_id}")
            
            # Find and print the matched truth particle
            matched_truth_particle = None
            for reco_p, truth_p in data['particle_matches_r2t']:
                if reco_p == particle:  # Check if this reconstructed particle matches
                    matched_truth_particle = truth_p
                    break
            
            # Print the matched truth particle if found
            if matched_truth_particle:
                print(f"TruthParticle:    {matched_truth_particle}\n")
            else:
                print("TruthParticle: None\n")
    
    print(f"Total Primary Particles Within Limits for Interaction ID {target_reco_interaction_id}: {len(filtered_primary_particles)}")


Found target event 79 at entry 72
Total Reconstructed Particles for Event 79: 17
reco_particles[5]: RecoParticle(ID: 5   | PID: Muon     | Primary: 0  | Size: 21    | Match: 1  ) | Reco Interaction ID: 2
TruthParticle:    TruthParticle(ID: 1   | PID: Pion     | Primary: 0  | Size: 21    | Match: 5  )

reco_particles[7]: RecoParticle(ID: 7   | PID: Pion     | Primary: 0  | Size: 36    | Match: 2  ) | Reco Interaction ID: 2
TruthParticle:    TruthParticle(ID: 2   | PID: Pion     | Primary: 0  | Size: 36    | Match: 7  )

Total Primary Particles Within Limits for Interaction ID 2: 2


### III. Filtering Reconstructed Particles for Target Interaction ID Within Spatial Limits

The script locates the target reconstructed interaction ID (`target_reco_interaction_id`) in the dataset, retrieves its entry, and filters **primary reconstructed and truth particles** that:
- Belong to the specified interaction ID.
- Are `primary`.
- Are within defined spatial limits (`minX`, `maxX`, `minY`, `maxY`, `minZ`, `maxZ`).
- A `length` selection can also be applied and printed.

It prints details of reconstructed particles and their interaction IDs, along with the total count of filtered particles.

In [8]:
import h5py

# Define the target reconstructed interaction ID
target_reco_interaction_id = 2  # Example: Target Interaction ID

# Define spatial limits
distance_from_wall = 5.0
minX = -63.931 + distance_from_wall
maxX = +63.931 - distance_from_wall
minY = -62.076 + distance_from_wall
maxY = +62.076 - distance_from_wall
minZ = -64.538 + distance_from_wall
maxZ = +64.538 - distance_from_wall

# Minimum length cut in cm
length_cut = 2.0

# Locate the entry corresponding to the target event
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    target_entry = None
    
    for idx, (run, subrun, event) in enumerate(run_info):
        if event == target_event:
            target_entry = idx
            print(f"Found target event {target_event} at entry {idx}")
            break
    else:
        print(f"Event {target_event} not found in run_info.")
        target_entry = None

# Process the entry if found
if target_entry is not None:
    data = driver.process(entry=target_entry)
    reco_particles = data['reco_particles']
    print(f"Total Reconstructed Particles for Event {target_event}: {len(reco_particles)}")
    
    # Filter and print particles within limits, primary, and belonging to the target interaction ID
    filtered_primary_particles = []
    for i, particle in enumerate(reco_particles):
        # Access start_point for spatial coordinates using dot notation
        start_point = particle.start_point
        x, y, z = start_point
        
        # Check if the particle is primary and matches the target interaction ID
        is_primary = particle.is_primary
        interaction_id = particle.interaction_id  # Get the interaction ID
        length = particle.length  # Length attribute of the particle
        
        # Apply spatial limits, primary condition, and target interaction ID
        if (
            # length > length_cut and
            # is_primary and
            # minX < x < maxX and
            # minY < y < maxY and
            # minZ < z < maxZ and
            interaction_id == target_reco_interaction_id
        ):
            filtered_primary_particles.append(particle)
            print(f"reco_particles[{i}]: {particle} | Reco Interaction ID: {interaction_id}")
            # print(f"reco_particles[{i}]: {particle} | Reco Interaction ID: {interaction_id} | Length: {length:.2f} cm")
            
            # Find and print the matched truth particle
            matched_truth_particle = None
            for reco_p, truth_p in data['particle_matches_r2t']:
                if reco_p == particle:  # Check if this reconstructed particle matches
                    matched_truth_particle = truth_p
                    break
            
            # Print the matched truth particle if found
            if matched_truth_particle:
                print(f"TruthParticle:    {matched_truth_particle}\n")
                # print(f"TruthParticle:    {matched_truth_particle} | Length: {length:.2f} cm\n")
            else:
                print("TruthParticle: None\n")
    
    print(f"Total Primary Particles Within Limits for Interaction ID {target_reco_interaction_id}: {len(filtered_primary_particles)}")

Found target event 79 at entry 72
Total Reconstructed Particles for Event 79: 17
reco_particles[5]: RecoParticle(ID: 5   | PID: Muon     | Primary: 0  | Size: 21    | Match: 1  ) | Reco Interaction ID: 2
TruthParticle:    TruthParticle(ID: 1   | PID: Pion     | Primary: 0  | Size: 21    | Match: 5  )

reco_particles[7]: RecoParticle(ID: 7   | PID: Pion     | Primary: 0  | Size: 36    | Match: 2  ) | Reco Interaction ID: 2
TruthParticle:    TruthParticle(ID: 2   | PID: Pion     | Primary: 0  | Size: 36    | Match: 7  )

Total Primary Particles Within Limits for Interaction ID 2: 2


### IV. Filtered Particle Visualization

This script visualizes and saves plots of filtered reconstructed particles and interactions. 
   - Particle plots are saved as:
     - **HTML**: `MR6p1_<file_segment>_entry_<ENTRY>_event_<target_event>_int_<target_reco_interaction_id>_particles.html`
     - **PNG**: `MR6p1_<file_segment>_entry_<ENTRY>_event_<target_event>_int_<target_reco_interaction_id>_particles.png`

In [ ]:
from spine.vis.out import Drawer
import plotly.io as pio
import os

# Extract the file number (e.g., '0000000') from DATA_PATH
file_number = os.path.basename(DATA_PATH).split('.')[3]

# Extract IDs of the filtered primary particles and their interaction IDs
filtered_particle_ids = [particle.id for particle in filtered_primary_particles]
matched_truth_particle_ids = set()

# Match truth particle IDs
for reco_particle in filtered_primary_particles:
    for reco_p, truth_p in data['particle_matches_r2t']:
        if reco_p.id == reco_particle.id:
            matched_truth_particle_ids.add(truth_p.id)

# Filter truth interactions
filtered_truth_interactions = [
    interaction for interaction in data['truth_interactions'] 
    if any(particle.id in matched_truth_particle_ids for particle in interaction.particles)
]

# Create a Drawer instance
drawer = Drawer(data, draw_mode='both', detector='2x2', split_scene=True)

# Pre-filter the data
data['reco_particles'] = [p for p in data['reco_particles'] if p.id in filtered_particle_ids]
data['truth_particles'] = [p for p in data['truth_particles'] if p.id in matched_truth_particle_ids]
data['truth_interactions'] = filtered_truth_interactions

# Generate the visualization
fig = drawer.get(
    obj_type='particles',
    attr='pid',
    draw_end_points=True,
    draw_vertices=True,
    synchronize=True,
    titles=['Reconstructed Particles', 'Truth Particles'],
    split_traces=True
)

# Adjust the figure layout for better visibility
fig.update_layout(
    height=800,
    width=1000,
)

# Construct dynamic filenames
html_filename = f"MR6p1_{file_number}_entry_{ENTRY}_event_{target_event}_int_{target_reco_interaction_id}.html"
png_filename = f"MR6p1_{file_number}_entry_{ENTRY}_event_{target_event}_int_{target_reco_interaction_id}.png"

# Save as HTML
pio.write_html(fig, file=html_filename)
print(f"Visualization saved as HTML: {html_filename}")

# Save as PNG
pio.write_image(fig, file=png_filename, format='png', width=1000, height=800)
print(f"Visualization saved as PNG: {png_filename}")

# Display the visualization
fig.show()

In [10]:
pip install tabulate


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import sys
import numpy as np
import h5py
from tabulate import tabulate  # Install using: pip install tabulate
from spine.driver import Driver
from spine.utils.globals import PID_LABELS  # Ensure PID_LABELS maps numbers to particle names

print('Number of entries in the loader:', len(driver))

# Open the HDF5 file to read event details
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    total_entries = len(run_info)

    # Initialize results storage
    all_table_data = []

    for ENTRY in range(total_entries):
        # Get the target event from run_info
        try:
            run, subrun, target_event = run_info[ENTRY]  # Extract event number
        except IndexError:
            print(f"Skipping entry {ENTRY} due to missing run_info.")
            continue

        # Process data for the current entry
        try:
            data = driver.process(entry=ENTRY)
            reco_particles = data['reco_particles']
            truth_particles = data['truth_particles']
        except Exception as e:
            print(f"Skipping entry {ENTRY} due to error: {e}")
            continue  # Skip problematic entries

        # Count reco and truth particles per interaction
        reco_interaction_particle_counts = {}
        truth_interaction_particle_counts = {}

        # Initialize particle type counts per interaction
        reco_pid_counts = {}
        truth_pid_counts = {}

        # Process reco particles
        for particle in reco_particles:
            interaction_id = particle.interaction_id
            pid = particle.pid

            # Update reco interaction count
            reco_interaction_particle_counts[interaction_id] = reco_interaction_particle_counts.get(interaction_id, 0) + 1
            
            # Count muons, pions, and protons per interaction
            if interaction_id not in reco_pid_counts:
                reco_pid_counts[interaction_id] = {2: 0, 3: 0, 4: 0}  # Muon, Pion, Proton

            if pid in reco_pid_counts[interaction_id]:
                reco_pid_counts[interaction_id][pid] += 1

        # Process truth particles
        for particle in truth_particles:
            interaction_id = particle.interaction_id
            pid = particle.pid

            # Update truth interaction count
            truth_interaction_particle_counts[interaction_id] = truth_interaction_particle_counts.get(interaction_id, 0) + 1

            # Count muons, pions, and protons per interaction
            if interaction_id not in truth_pid_counts:
                truth_pid_counts[interaction_id] = {2: 0, 3: 0, 4: 0}  # Muon, Pion, Proton

            if pid in truth_pid_counts[interaction_id]:
                truth_pid_counts[interaction_id][pid] += 1

        # Collect interaction IDs present in either reco or truth
        interaction_ids = sorted(set(reco_interaction_particle_counts.keys()).union(truth_interaction_particle_counts.keys()))

        # Store table data for this entry
        for interaction_id in interaction_ids:
            reco_count = reco_interaction_particle_counts.get(interaction_id, 0)
            truth_count = truth_interaction_particle_counts.get(interaction_id, 0)

            # Get muon, pion, proton counts (default to 0 if not found)
            reco_muons = reco_pid_counts.get(interaction_id, {}).get(2, 0)
            reco_pions = reco_pid_counts.get(interaction_id, {}).get(3, 0)
            reco_protons = reco_pid_counts.get(interaction_id, {}).get(4, 0)

            truth_muons = truth_pid_counts.get(interaction_id, {}).get(2, 0)
            truth_pions = truth_pid_counts.get(interaction_id, {}).get(3, 0)
            truth_protons = truth_pid_counts.get(interaction_id, {}).get(4, 0)

            # Format counts as "Reco/Truth"
            muon_str = f"{reco_muons}/{truth_muons}"
            pion_str = f"{reco_pions}/{truth_pions}"
            proton_str = f"{reco_protons}/{truth_protons}"

            # Append row to table
            all_table_data.append([
                ENTRY, target_event, interaction_id, 
                reco_count, truth_count, 
                muon_str, pion_str, proton_str
            ])

# Print final tabulated output
print("\nSummary of Particles per Interaction ID for All Entries:")
print(tabulate(all_table_data, headers=[
    "Entry", "Event ID", "Interaction ID", 
    "Reco Particles", "Truth Particles", 
    "Muons (R/T)", "Pions (R/T)", "Protons (R/T)"
], tablefmt="pretty"))


Number of entries in the loader: 172

Summary of Particles per Interaction ID for All Entries:
+-------+----------+----------------+----------------+-----------------+-------------+-------------+---------------+
| Entry | Event ID | Interaction ID | Reco Particles | Truth Particles | Muons (R/T) | Pions (R/T) | Protons (R/T) |
+-------+----------+----------------+----------------+-----------------+-------------+-------------+---------------+
|   0   |    0     |       0        |       16       |       21        |     3/1     |     3/4     |     4/10      |
|   2   |    2     |       0        |       2        |        2        |     1/1     |     1/1     |      0/0      |
|   3   |    3     |       0        |       0        |        1        |     0/0     |     0/0     |      0/1      |
|   4   |    4     |       0        |       1        |        2        |     0/1     |     0/0     |      0/0      |
|   4   |    4     |       1        |       1        |        1        |     1/1     |

In [12]:
import sys
import numpy as np
import h5py
import os
import plotly.io as pio
from tabulate import tabulate  # Install using: pip install tabulate
from spine.driver import Driver
from spine.utils.globals import PID_LABELS  # Ensure PID_LABELS maps numbers to particle names
from spine.vis.out import Drawer

# Define spatial limits
distance_from_wall = 5.0
minX = -63.931 + distance_from_wall
maxX = +63.931 - distance_from_wall
minY = -62.076 + distance_from_wall
maxY = +62.076 - distance_from_wall
minZ = -64.538 + distance_from_wall
maxZ = +64.538 - distance_from_wall

# Open the HDF5 file to read event details
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    total_entries = len(run_info)

    for ENTRY in range(total_entries):
        # Get the target event from run_info
        try:
            run, subrun, target_event = run_info[ENTRY]  # Extract event number
        except IndexError:
            print(f"Skipping entry {ENTRY} due to missing run_info.")
            continue

        # Process data for the current entry
        try:
            data = driver.process(entry=ENTRY)
            reco_particles = data['reco_particles']
            truth_particles = data['truth_particles']
        except Exception as e:
            print(f"Skipping entry {ENTRY} due to error: {e}")
            continue  # Skip problematic entries

        # Extract all unique interaction IDs
        interaction_ids = sorted(set(particle.interaction_id for particle in truth_particles))
        print(f"\nEntry {ENTRY} contains {len(interaction_ids)} interactions: {interaction_ids}")

        # Collect all particles for visualization
        all_reco_particles = []
        all_truth_particles = []
        matched_truth_particle_ids = set()
        selected_interaction_ids = set()  # Stores interactions that pass the selection criteria

        for TARGET_INTERACTION_ID in interaction_ids:
            # Get particles for the current interaction
            reco_particles = [p for p in data['reco_particles'] if p.interaction_id == TARGET_INTERACTION_ID]
            truth_particles = [p for p in data['truth_particles'] if p.interaction_id == TARGET_INTERACTION_ID]
            
            # Apply selection criteria for truth particles
            passes_selection = False
            for particle in truth_particles:
                if particle.is_primary:
                    x, y, z = particle.start_point  # Extract start point coordinates
                    if minX < x < maxX and minY < y < maxY and minZ < z < maxZ:
                        passes_selection = True
                        break  # If one truth particle passes, we keep the entire interaction
            
            if not passes_selection:
                continue  # Skip interactions that don't satisfy the spatial constraints
            
            # If it passes, add its particles for visualization
            selected_interaction_ids.add(TARGET_INTERACTION_ID)
            all_reco_particles.extend(reco_particles)
            all_truth_particles.extend(truth_particles)

            # Match truth particle IDs with reco
            for reco_particle in all_reco_particles:
                for reco_p, truth_p in data['particle_matches_r2t']:
                    if reco_p and truth_p:  # Ensure neither reco_p nor truth_p is None
                        if reco_p.id == reco_particle.id:
                            matched_truth_particle_ids.add(truth_p.id)

        # If no interactions pass the selection criteria, skip visualization
        if not selected_interaction_ids:
            print(f"Skipping Entry {ENTRY}: No valid interactions passed spatial selection.")
            continue

        # Filter truth interactions to include only selected ones
        filtered_truth_interactions = [
            interaction for interaction in data['truth_interactions']
            if interaction.id in selected_interaction_ids
        ]
        
        # Create a Drawer instance
        drawer = Drawer(data, draw_mode='both', detector='2x2', split_scene=True)
        
        # Pre-filter the data for visualization
        data['reco_particles'] = all_reco_particles
        data['truth_particles'] = [p for p in all_truth_particles if p.id in matched_truth_particle_ids]
        data['truth_interactions'] = filtered_truth_interactions
        
        # Generate the visualization
        fig = drawer.get(
            obj_type='particles',
            attr='pid',
            draw_end_points=True,
            draw_vertices=True,
            synchronize=True,
            titles=[f"Reconstructed Event", f"Truth Event"],
            split_traces=True
        )
        
        # Adjust the figure layout for better visibility
        fig.update_layout(height=800, width=1000)
        
        # Extract the file number (e.g., '0000000') from DATA_PATH
        file_number = os.path.basename(DATA_PATH).split('.')[3]
        
        # Construct dynamic filenames
        html_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.html"
        png_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.png"
        
        # Save visualization as HTML
        pio.write_html(fig, file=html_filename)
        print(f"\nVisualization saved as HTML: {html_filename}")
        
        # Save visualization as PNG
        pio.write_image(fig, file=png_filename, format='png', width=1000, height=800)
        print(f"Visualization saved as PNG: {png_filename}")


Entry 0 contains 1 interactions: [0]

Visualization saved as HTML: MR6p1_0000000_entry_0_all_interactions.html
Visualization saved as PNG: MR6p1_0000000_entry_0_all_interactions.png

Entry 1 contains 0 interactions: []
Skipping Entry 1: No valid interactions passed spatial selection.

Entry 2 contains 1 interactions: [0]
Skipping Entry 2: No valid interactions passed spatial selection.

Entry 3 contains 1 interactions: [0]
Skipping Entry 3: No valid interactions passed spatial selection.

Entry 4 contains 2 interactions: [0, 1]
Skipping Entry 4: No valid interactions passed spatial selection.

Entry 5 contains 0 interactions: []
Skipping Entry 5: No valid interactions passed spatial selection.

Entry 6 contains 2 interactions: [0, 1]
Skipping Entry 6: No valid interactions passed spatial selection.

Entry 7 contains 1 interactions: [0]
Skipping Entry 7: No valid interactions passed spatial selection.

Entry 8 contains 2 interactions: [0, 1]
Skipping Entry 8: No valid interactions pass

In [ ]:
from spine.vis.out import Drawer
import plotly.io as pio
import os
from tabulate import tabulate  # Install via: pip install tabulate

# === User Input: Provide the Entry Number ===
ENTRY = 40  # Change this to the desired entry number

# Extract the file number (e.g., '0000000') from DATA_PATH
file_number = os.path.basename(DATA_PATH).split('.')[3]

# Load data for the given entry
data = driver.process(entry=ENTRY)

# Extract all unique interaction IDs
interaction_ids = sorted(set(particle.interaction_id for particle in data['truth_particles']))
print(f"\nEntry {ENTRY} contains {len(interaction_ids)} interactions: {interaction_ids}")

# Initialize summary storage
interaction_summary = []

# Include all reco and truth particles from all interactions
all_reco_particles = []
all_truth_particles = []

# Define particle types to count
PARTICLE_LABELS = {
    0: 'Photon', 1: 'Electron', 2: 'Muon', 3: 'Pion', 4: 'Proton', 5: 'Kaon'
}
particle_ids = list(PARTICLE_LABELS.keys())  # [0, 1, 2, 3, 4, 5]

for TARGET_INTERACTION_ID in interaction_ids:
    # Filter particles for the current interaction
    reco_particles = [p for p in data['reco_particles'] if p.interaction_id == TARGET_INTERACTION_ID]
    truth_particles = [p for p in data['truth_particles'] if p.interaction_id == TARGET_INTERACTION_ID]

    # Count different particle types
    reco_counts = {pid: 0 for pid in particle_ids}
    truth_counts = {pid: 0 for pid in particle_ids}

    for p in reco_particles:
        if p.pid in reco_counts:
            reco_counts[p.pid] += 1

    for p in truth_particles:
        if p.pid in truth_counts:
            truth_counts[p.pid] += 1

    # Append to summary table
    interaction_summary.append([
        ENTRY, TARGET_INTERACTION_ID, len(reco_particles), len(truth_particles),
        f"{reco_counts[0]}/{truth_counts[0]}",  # Photons (Reco/Truth)
        f"{reco_counts[1]}/{truth_counts[1]}",  # Electrons (Reco/Truth)
        f"{reco_counts[2]}/{truth_counts[2]}",  # Muons (Reco/Truth)
        f"{reco_counts[3]}/{truth_counts[3]}",  # Pions (Reco/Truth)
        f"{reco_counts[4]}/{truth_counts[4]}",  # Protons (Reco/Truth)
        f"{reco_counts[5]}/{truth_counts[5]}"   # Kaons (Reco/Truth)
    ])

    # Append particles for visualization
    all_reco_particles.extend(reco_particles)
    all_truth_particles.extend(truth_particles)

# Print interaction summary
print("\nSummary of Each Interaction in Entry:")
print(tabulate(interaction_summary, headers=[
    "Entry", "Interaction ID", "Reco Particles", "Truth Particles",
    "Photons (R/T)", "Electrons (R/T)", "Muons (R/T)",
    "Pions (R/T)", "Protons (R/T)", "Kaons (R/T)"
], tablefmt="pretty"))

# Extract particle IDs
filtered_reco_particle_ids = [p.id for p in all_reco_particles]
filtered_truth_particle_ids = [p.id for p in all_truth_particles]

# Match truth particle IDs with reco
matched_truth_particle_ids = set()
for reco_particle in all_reco_particles:
    for reco_p, truth_p in data['particle_matches_r2t']:
        if reco_p.id == reco_particle.id:
            matched_truth_particle_ids.add(truth_p.id)

# Filter truth interactions to include only the relevant ones
filtered_truth_interactions = [
    interaction for interaction in data['truth_interactions']
    if interaction.id in interaction_ids
]

# Create a Drawer instance
drawer = Drawer(data, draw_mode='both', detector='2x2', split_scene=True)

# Pre-filter the data for visualization
data['reco_particles'] = all_reco_particles
data['truth_particles'] = [p for p in all_truth_particles if p.id in matched_truth_particle_ids]
data['truth_interactions'] = filtered_truth_interactions

# Generate the visualization
fig = drawer.get(
    obj_type='particles',
    attr='pid',
    draw_end_points=True,
    draw_vertices=True,
    synchronize=True,
    titles=[f"Reconstructed Event",
            f"Truth Event"],
    split_traces=True
)

# Adjust the figure layout for better visibility
fig.update_layout(
    height=800,
    width=1000,
)

# Construct dynamic filenames
html_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.html"
png_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.png"

# Save visualization as HTML
pio.write_html(fig, file=html_filename)
print(f"\nVisualization saved as HTML: {html_filename}")

# Save visualization as PNG
pio.write_image(fig, file=png_filename, format='png', width=1000, height=800)
print(f"Visualization saved as PNG: {png_filename}")

# Display the visualization
fig.show()


In [ ]:
import sys
import numpy as np
import h5py
import os
import plotly.io as pio
from tabulate import tabulate  # Install using: pip install tabulate
from spine.driver import Driver
from spine.utils.globals import PID_LABELS  # Ensure PID_LABELS maps numbers to particle names
from spine.vis.out import Drawer

# Open the HDF5 file to read event details
with h5py.File(DATA_PATH, 'r') as f:
    run_info = f['run_info']
    total_entries = len(run_info)

    for ENTRY in range(total_entries):
        # Get the target event from run_info
        try:
            run, subrun, target_event = run_info[ENTRY]  # Extract event number
        except IndexError:
            print(f"Skipping entry {ENTRY} due to missing run_info.")
            continue

        # Process data for the current entry
        try:
            data = driver.process(entry=ENTRY)
            reco_particles = data['reco_particles']
            truth_particles = data['truth_particles']
        except Exception as e:
            print(f"Skipping entry {ENTRY} due to error: {e}")
            continue  # Skip problematic entries

        # Extract all unique interaction IDs
        interaction_ids = sorted(set(particle.interaction_id for particle in truth_particles))
        print(f"\nEntry {ENTRY} contains {len(interaction_ids)} interactions: {interaction_ids}")

        # Collect all particles for visualization
        all_reco_particles = []
        all_truth_particles = []
        matched_truth_particle_ids = set()

        for TARGET_INTERACTION_ID in interaction_ids:
            reco_particles = [p for p in data['reco_particles'] if p.interaction_id == TARGET_INTERACTION_ID]
            truth_particles = [p for p in data['truth_particles'] if p.interaction_id == TARGET_INTERACTION_ID]
            
            all_reco_particles.extend(reco_particles)
            all_truth_particles.extend(truth_particles)
            
            # Match truth particle IDs with reco
            for reco_particle in all_reco_particles:
                for reco_p, truth_p in data['particle_matches_r2t']:
                    if reco_p and truth_p:  # Ensure neither reco_p nor truth_p is None
                        if reco_p.id == reco_particle.id:
                            matched_truth_particle_ids.add(truth_p.id)

        
        # Filter truth interactions to include only relevant ones
        filtered_truth_interactions = [
            interaction for interaction in data['truth_interactions']
            if interaction.id in interaction_ids
        ]
        
        # Create a Drawer instance
        drawer = Drawer(data, draw_mode='both', detector='2x2', split_scene=True)
        
        # Pre-filter the data for visualization
        data['reco_particles'] = all_reco_particles
        data['truth_particles'] = [p for p in all_truth_particles if p.id in matched_truth_particle_ids]
        data['truth_interactions'] = filtered_truth_interactions
        
        # Generate the visualization
        fig = drawer.get(
            obj_type='particles',
            attr='pid',
            draw_end_points=True,
            draw_vertices=True,
            synchronize=True,
            titles=[f"Reconstructed Event", f"Truth Event"],
            split_traces=True
        )
        
        # Adjust the figure layout for better visibility
        fig.update_layout(height=800, width=1000)
        
        # Extract the file number (e.g., '0000000') from DATA_PATH
        file_number = os.path.basename(DATA_PATH).split('.')[3]
        
        # Construct dynamic filenames
        html_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.html"
        png_filename = f"MR6p1_{file_number}_entry_{ENTRY}_all_interactions.png"
        
        # Save visualization as HTML
        pio.write_html(fig, file=html_filename)
        print(f"\nVisualization saved as HTML: {html_filename}")
        
        # Save visualization as PNG
        pio.write_image(fig, file=png_filename, format='png', width=1000, height=800)
        print(f"Visualization saved as PNG: {png_filename}")



Entry 0 contains 1 interactions: [0]

Visualization saved as HTML: MR6p1_0000001_entry_0_all_interactions.html
